In [1]:
import yfinance as yf
import pandas as pd
import time
import logging

In [2]:
logging.basicConfig(
    filename="data_fetch.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [3]:
TICKERS = [
    "HDFCBANK.NS",   # Banking
    "TCS.NS",        # IT
    "HINDUNILVR.NS", # FMCG
    "RELIANCE.NS",   # Energy
    "MARUTI.NS",     # Automobile
    "SUNPHARMA.NS",  # Pharma
    "TATASTEEL.NS",  # Metals
    "BHARTIARTL.NS", # Telecom
    "LT.NS",         # Infrastructure
    "DMART.NS",      # Retail
    "^NSEI"          # Index
]

In [4]:
START_DATE = "2018-01-01"
END_DATE = "2023-12-31"

In [14]:
def fetch_with_retry(ticker, retries=3, delay=3):
    for attempt in range(retries):
        try:
            data = yf.download(
                ticker,
                start=START_DATE,
                end=END_DATE,
                progress=False,
                auto_adjust=True
            )

            if data.empty:
                raise ValueError("Empty data returned")

            logging.info(f"Success: {ticker}")
            return data

        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed for {ticker}: {e}")
            time.sleep(delay)

    logging.error(f"Failed to fetch data for {ticker}")
    return None

In [6]:
def validate_data(df, ticker):
    if df is None:
        return False

    if df.isnull().sum().sum() > 0:
        logging.warning(f"Missing values found in {ticker}")

    return True

In [7]:
import os
os.makedirs("data", exist_ok=True)

def save_data(df, ticker):
    file_path = f"data/{ticker}.csv"
    df.to_csv(file_path)
    logging.info(f"Saved: {file_path}")

In [8]:
all_data = {}

for ticker in TICKERS:
    print(f"Fetching {ticker}...")

    df = fetch_with_retry(ticker)

    if validate_data(df, ticker):
        save_data(df, ticker)
        all_data[ticker] = df

    # Handle API rate limits
    time.sleep(2)

print("✅ Data collection completed")

Fetching HDFCBANK.NS...
Fetching TCS.NS...
Fetching HINDUNILVR.NS...
Fetching RELIANCE.NS...
Fetching MARUTI.NS...
Fetching SUNPHARMA.NS...
Fetching TATASTEEL.NS...
Fetching BHARTIARTL.NS...
Fetching LT.NS...
Fetching DMART.NS...
Fetching ^NSEI...
✅ Data collection completed


In [9]:
import os
print(os.getcwd())

C:\Users\Akshaya\sql_notebook


In [15]:
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

In [16]:
df['Returns'] = df['Close'].pct_change()

In [17]:
print(df.head())

Price              Close          High           Low          Open  Volume  \
Date                                                                         
2018-01-02  10442.200195  10495.200195  10404.650391  10477.549805  153400   
2018-01-03  10443.200195  10503.599609  10429.549805  10482.650391  167300   
2018-01-04  10504.799805  10513.000000  10441.450195  10469.400391  174900   
2018-01-05  10558.849609  10566.099609  10520.099609  10534.250000  180900   
2018-01-08  10623.599609  10631.200195  10588.549805  10591.700195  169000   

Price        Returns  
Date                  
2018-01-02       NaN  
2018-01-03  0.000096  
2018-01-04  0.005899  
2018-01-05  0.005145  
2018-01-08  0.006132  
